# Guidance Pattern Viewer

Reads an ISOXML v4 TaskData file, extracts all guidance patterns from every
partfield, converts them to Shapely geometries, and displays them on an
interactive Folium map.

**Prerequisites:** `pip install isoxml[viz]`

In [ ]:
from pathlib import Path

import geopandas as gpd

from isoxml.geometry import ShapelyConverterV4
from isoxml.io import read_from_path

# Use any TASKDATA.XML path here
from tests.resources.test_resources import TEST_RES_DIR
TASKDATA_PATH = TEST_RES_DIR / "isoxml/v4/cnh_export/TASKDATA.XML"

## Load TaskData

In [ ]:
task_data, _refs = read_from_path(TASKDATA_PATH)

print(f"Partfields:  {len(task_data.partfields)}")
for pfd in task_data.partfields:
    n_groups = len(pfd.guidance_groups)
    n_patterns = sum(len(gg.guidance_patterns) for gg in pfd.guidance_groups)
    print(f"  PFD '{pfd.designator}': {n_groups} group(s), {n_patterns} pattern(s)")

## Extract Guidance Patterns

In [ ]:
converter = ShapelyConverterV4()

records = []
for pfd in task_data.partfields:
    for gg in pfd.guidance_groups:
        for gp in gg.guidance_patterns:
            for lsg in gp.line_strings:
                if len(lsg.points) < 2:
                    continue
                records.append({
                    "partfield": pfd.designator,
                    "group": gg.designator,
                    "pattern_id": gp.id,
                    "pattern_type": str(gp.type),
                    "designator": gp.designator,
                    "n_points": len(lsg.points),
                    "geometry": converter.to_shapely_geom(lsg),
                })

gdf = gpd.GeoDataFrame(records, crs="EPSG:4326")
print(f"Extracted {len(gdf)} guidance line string(s)")
gdf.drop(columns="geometry")

## Interactive Map

In [ ]:
m = gdf.explore(
    column="pattern_type",
    tooltip=["designator", "pattern_type", "n_points"],
    popup=True,
    style_kwds={"weight": 3},
    legend=True,
    legend_kwds={"caption": "Guidance Pattern Type"},
)
m

## Partfield Boundaries (optional overlay)

In [ ]:
boundary_records = []
for pfd in task_data.partfields:
    for poly in pfd.polygons:
        boundary_records.append({
            "designator": pfd.designator,
            "geometry": converter.to_shapely_polygon(poly),
        })

if boundary_records:
    pfd_gdf = gpd.GeoDataFrame(boundary_records, crs="EPSG:4326")
    m = pfd_gdf.explore(
        m=m,
        color="gray",
        style_kwds={"fillOpacity": 0.1, "weight": 2},
        tooltip="designator",
    )
else:
    print("No partfield boundaries found.")
m